In [ ]:
!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
langchain-groq \
faiss-cpu \
pypdf \
sentence-transformers

In [1]:
import os

from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

print("✅ All libraries imported successfully!")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Retrieve the API key from Colab secrets and set it as an environment variable
from google.colab import userdata

try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("GROQ_API_KEY successfully loaded from Colab secrets.")
except userdata.exceptions.KeyNotFoundError:
    print("GROQ_API_KEY not found in Colab secrets. Please add it to your secrets manager.")
except Exception as e:
    print(f"An error occurred while loading GROQ_API_KEY: {e}")

GROQ_API_KEY successfully loaded from Colab secrets.


## **Initialize the LLM**

We'll use Llama 3.3 70B Versatile because it's one of the best models available on Groq for RAG tasks.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("✅ LLM Initialized Successfully")

✅ LLM Initialized Successfully


### **Test the LLM**

In [ ]:
response = llm.invoke("What is Retrieval-Augmented Generation (RAG)?")

print(response.content)

Retrieval-Augmented Generation (RAG) is a type of natural language processing (NLP) and generation technique that combines the strengths of retrieval-based and generation-based approaches. It aims to improve the quality and accuracy of generated text by leveraging the power of retrieval and generation models.

In traditional generation models, the system generates text from scratch based on the input prompt or context. However, this approach can lead to issues such as:

1. Lack of context: The model may not have enough information to generate accurate and relevant text.
2. Limited knowledge: The model's training data may not cover all possible topics, entities, or concepts.
3. Hallucinations: The model may generate text that is not grounded in reality or is factually incorrect.

RAG addresses these limitations by incorporating a retrieval step into the generation process. The system first retrieves relevant information from a large corpus of text, such as a database or a knowledge grap

# **📍Step 1: PDF Upload & Loading**
# 🎯 Goal

By the end of this step, you'll understand:

*   How to upload a PDF
*   How LangChain reads PDFs
*   What a Document object is
*   Why metadata is important
*   How to inspect your data before processing it

PDF
 │
 ▼
PDF Loader
 │
 ▼
Extract Text
 │
 ▼
LangChain Documents

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Classroom Attendance Systems Based on Bluetooth Low Energy Indoor Positioning Technology for Smart Campus (1).pdf to Classroom Attendance Systems Based on Bluetooth Low Energy Indoor Positioning Technology for Smart Campus (1) (1).pdf


In [ ]:
# Get the uploaded filename

pdf_file = list(uploaded.keys())[0]

print("Uploaded PDF:", pdf_file)

Uploaded PDF: Classroom Attendance Systems Based on Bluetooth Low Energy Indoor Positioning Technology for Smart Campus (1) (1).pdf


In [ ]:
# Get the uploaded filename

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_file)

documents = loader.load()

In [ ]:
# Check how many pages were loaded

print("Number of pages:", len(documents))

Number of pages: 18


In [ ]:
print(documents[0].page_content)

information
Article
Classroom Attendance Systems Based on Bluetooth
Low Energy Indoor Positioning Technology for
Smart Campus
Apiruk Puckdeevongs 1,*, N. K. Tripathi 1, Apichon Witayangkurn 1
and Poompat Saengudomlert 2
1 Remote Sensing and Geographic Information Systems Field of Study, School of Engineering and Technology,
Asian Institute of Technology, P .O. Box 4, Klong Luang, Pathumthani 12120, Thailand;
nitinkt@ait.asia (N.K.T.); apichon@ait.asia (A.W.)
2 Telecommunications Field of Study, School of Engineering and Technology, Asian Institute of Technology,
P .O. Box 4, Klong Luang, Pathumthani 12120, Thailand; poompat@gmail.com
* Correspondence: st106421@ait.asia or apiruk.pui@gmail.com; Tel.: +66-86896-3333
Received: 3 May 2020; Accepted: 16 June 2020; Published: 19 June 2020
/gid00030/gid00035/gid00032/gid00030/gid00038/gid00001/gid00033/gid00042/gid00045/gid00001
/gid00048/gid00043/gid00031/gid00028/gid00047/gid00032/gid00046
Abstract: Student attendance during classroom hours

In [ ]:
print(documents[0].metadata)

{'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2020-06-19T17:02:24+08:00', 'author': 'Apiruk Puckdeevongs, N. K. Tripathi, Apichon Witayangkurn and Poompat Saengudomlert', 'title': 'Classroom Attendance Systems Based on Bluetooth Low Energy Indoor Positioning Technology for Smart Campus', 'subject': 'Student attendance during classroom hours is important, because it impacts the academic performance of students. Consequently, several universities impose a minimum attendance percentage criterion for students to be allowed to attend examinations; therefore, recording student attendance is a vital task. Conventional methods for recording student attendance in the classroom, such as roll-call and sign-in, are an inefficient use of instruction time and only increase teachers’ workloads. In this study, we propose a Bluetooth Low Energy-based student positioning framework for automatically recording student attendance in classrooms. The proposed archi

In [ ]:
for i in range(3):
    print("=" * 80)
    print(f"Page: {documents[i].metadata['page']}")
    print("=" * 80)
    print(documents[i].page_content[:500])
    print("\n")

Page: 0
information
Article
Classroom Attendance Systems Based on Bluetooth
Low Energy Indoor Positioning Technology for
Smart Campus
Apiruk Puckdeevongs 1,*, N. K. Tripathi 1, Apichon Witayangkurn 1
and Poompat Saengudomlert 2
1 Remote Sensing and Geographic Information Systems Field of Study, School of Engineering and Technology,
Asian Institute of Technology, P .O. Box 4, Klong Luang, Pathumthani 12120, Thailand;
nitinkt@ait.asia (N.K.T.); apichon@ait.asia (A.W.)
2 Telecommunications Field of Study, 


Page: 1
Information 2020, 11, 329 2 of 18
classrooms, which instructors manually perform. However, this traditional process of attendance
recording suﬀers from drawbacks and is prone to errors. In recent years, systems have been developed
to automate this process in classrooms using various technologies to save relevant instruction time and
improve the reliability of attendance reports. The biometric systems [7,8] have been widely proposed,
but they still have limitations in terms of 

# **Step 2: Chunking**

PDF

↓

Chunk 1

↓

Chunk 2

↓

Chunk 3

↓

Chunk 4

## **Now, when the user asks a question:**

Question

↓

Retriever

↓

Chunk 17

↓

LLM

↓

Answer

In [ ]:
# Create the Text Splitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

print("✅ Text Splitter Created Successfully")

✅ Text Splitter Created Successfully


In [ ]:
# Split the Documents
chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 100


In [ ]:
# Inspect the First Chunk
chunks[0]

Document(metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2020-06-19T17:02:24+08:00', 'author': 'Apiruk Puckdeevongs, N. K. Tripathi, Apichon Witayangkurn and Poompat Saengudomlert', 'title': 'Classroom Attendance Systems Based on Bluetooth Low Energy Indoor Positioning Technology for Smart Campus', 'subject': 'Student attendance during classroom hours is important, because it impacts the academic performance of students. Consequently, several universities impose a minimum attendance percentage criterion for students to be allowed to attend examinations; therefore, recording student attendance is a vital task. Conventional methods for recording student attendance in the classroom, such as roll-call and sign-in, are an inefficient use of instruction time and only increase teachers’ workloads. In this study, we propose a Bluetooth Low Energy-based student positioning framework for automatically recording student attendance in classrooms. 

In [ ]:
# Check Chunk Metadata
print(chunks[0].metadata)

{'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2020-06-19T17:02:24+08:00', 'author': 'Apiruk Puckdeevongs, N. K. Tripathi, Apichon Witayangkurn and Poompat Saengudomlert', 'title': 'Classroom Attendance Systems Based on Bluetooth Low Energy Indoor Positioning Technology for Smart Campus', 'subject': 'Student attendance during classroom hours is important, because it impacts the academic performance of students. Consequently, several universities impose a minimum attendance percentage criterion for students to be allowed to attend examinations; therefore, recording student attendance is a vital task. Conventional methods for recording student attendance in the classroom, such as roll-call and sign-in, are an inefficient use of instruction time and only increase teachers’ workloads. In this study, we propose a Bluetooth Low Energy-based student positioning framework for automatically recording student attendance in classrooms. The proposed archi

In [ ]:
# View the First Few Chunks
for i in range(3):
    print("=" * 80)
    print(f"Chunk {i+1}")
    print(f"Page: {chunks[i].metadata['page']}")
    print("-" * 80)
    print(chunks[i].page_content[:500])
    print("\n")

Chunk 1
Page: 0
--------------------------------------------------------------------------------
information
Article
Classroom Attendance Systems Based on Bluetooth
Low Energy Indoor Positioning Technology for
Smart Campus
Apiruk Puckdeevongs 1,*, N. K. Tripathi 1, Apichon Witayangkurn 1
and Poompat Saengudomlert 2
1 Remote Sensing and Geographic Information Systems Field of Study, School of Engineering and Technology,
Asian Institute of Technology, P .O. Box 4, Klong Luang, Pathumthani 12120, Thailand;
nitinkt@ait.asia (N.K.T.); apichon@ait.asia (A.W.)
2 Telecommunications Field of Study, 


Chunk 2
Page: 0
--------------------------------------------------------------------------------
/gid00030/gid00035/gid00032/gid00030/gid00038/gid00001/gid00033/gid00042/gid00045/gid00001
/gid00048/gid00043/gid00031/gid00028/gid00047/gid00032/gid00046
Abstract: Student attendance during classroom hours is important, because it impacts the academic
performance of students. Consequently, several uni

# **🚀 Step 3: Embeddings**

In [ ]:
# !pip install langchain-huggingface
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding Model Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding Model Loaded Successfully


In [ ]:
text = "Artificial Intelligence is transforming healthcare."

embedding = embedding_model.embed_query(text)

print(type(embedding))
print(len(embedding))

<class 'list'>
384


In [ ]:
embedding[:10]

[0.04022699221968651,
 0.03933247551321983,
 0.05482863634824753,
 -0.00986447837203741,
 -0.0191482026129961,
 0.014894649386405945,
 -0.030346447601914406,
 0.034190833568573,
 0.0007948574493639171,
 -0.02408294938504696]

## **🚀 Step 4: FAISS (Vector Database)**

In [ ]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("✅ FAISS Vector Store Created Successfully")

✅ FAISS Vector Store Created Successfully


In [ ]:
print("Total Chunks:", len(chunks))

Total Chunks: 100


In [ ]:
query = "What is the main objective of this research paper?"

results = vector_store.similarity_search(query, k=3)

In [ ]:
for i, doc in enumerate(results):
    print("=" * 80)
    print(f"Result {i+1}")
    print(f"Page Number: {doc.metadata['page']}")
    print("-" * 80)
    print(doc.page_content[:500])
    print("\n")

Result 1
Page Number: 6
--------------------------------------------------------------------------------
Figure 4. Attendance monitoring system.
The sensors used in this system are referred to as Bluetooth stations (running on Raspberry Pi with
BLE), which are used to measure and collect the RSSI of the students’ devices from the four reference
points to calculate the students’ position in the classroom.
The Bluetooth station node is the BLE hardware that is installed in the classroom, which sends out
Bluetooth signals and it is referenced for position identiﬁcation of a particular student in the c


Result 2
Page Number: 12
--------------------------------------------------------------------------------
19 1.510 3.949 3.115 44 0.818 4.183 2.703 69 0.460 2.956 2.079
20 0.367 5.148 3.469 45 0.382 2.264 1.342
21 0.510 3.326 1.227 46 0.218 2.258 1.321
22 0.090 1.994 0.849 47 1.457 5.106 2.677
23 0.393 1.782 1.005 48 0.334 4.078 2.221
24 0.507 5.795 1.661 49 1.024 3.335 2.225
The proposed 

In [ ]:
queries = [
    "What problem does this paper solve?",
    "Explain the methodology.",
    "What are the results?",
    "What is the conclusion?"
]

for q in queries:
    print(f"\n🔹 Query: {q}")
    docs = vector_store.similarity_search(q, k=2)

    for d in docs:
        print(f"Page: {d.metadata['page']}")
        print(d.page_content[:200])
        print("-" * 50)


🔹 Query: What problem does this paper solve?
Page: 3
systems typically adopt either trilateration or ﬁngerprinting-based approaches.
2.2.2. Trilateration
The trilateration technique estimates the distance by translating RSSI. At least three Bluetooth
re
--------------------------------------------------
Page: 14
was in the class. Figure 13 illustrates the status (late/in class/absent) of a student’s attendance for each
session, from the beginning until the end of the record. The attendance patterns for each s
--------------------------------------------------

🔹 Query: Explain the methodology.
Page: 11
4.2. Computation of Student Positioning in Classroom
In this section, we deal with the metrology of the position calculation technique that was developed
on the experimental site of type I, by applyin
--------------------------------------------------
Page: 3
systems typically adopt either trilateration or ﬁngerprinting-based approaches.
2.2.2. Trilateration
The trilateration technique 

In [ ]:
vector_store.save_local("faiss_index")

print("✅ FAISS Index Saved Successfully")

✅ FAISS Index Saved Successfully


In [ ]:
from langchain_community.vectorstores import FAISS

loaded_vector_store = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

print("✅ FAISS Loaded Successfully")

✅ FAISS Loaded Successfully


In [ ]:
results = loaded_vector_store.similarity_search(
    "What is the main contribution of this paper?",
    k=2
)

for doc in results:
    print(doc.metadata)
    print(doc.page_content[:300])
    print("=" * 80)

{'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2020-06-19T17:02:24+08:00', 'author': 'Apiruk Puckdeevongs, N. K. Tripathi, Apichon Witayangkurn and Poompat Saengudomlert', 'title': 'Classroom Attendance Systems Based on Bluetooth Low Energy Indoor Positioning Technology for Smart Campus', 'subject': 'Student attendance during classroom hours is important, because it impacts the academic performance of students. Consequently, several universities impose a minimum attendance percentage criterion for students to be allowed to attend examinations; therefore, recording student attendance is a vital task. Conventional methods for recording student attendance in the classroom, such as roll-call and sign-in, are an inefficient use of instruction time and only increase teachers’ workloads. In this study, we propose a Bluetooth Low Energy-based student positioning framework for automatically recording student attendance in classrooms. The proposed archi

# **📍Current Architecture**
                 
                 Research Paper
                        │
                        ▼
                  PDF Loader ✅
                        │
                        ▼
                  Documents ✅
                        │
                        ▼
                 Text Chunking ✅
                        │
                        ▼
                  Embeddings ✅
                        │
                        ▼
                 FAISS Vector DB ✅
                        │
                        ▼
                 Similarity Search ✅

# **🚀 Step 5 — Retriever**

In [ ]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.7
    }
)

In [ ]:
query = "What is the main objective of this paper?"

retrieved_docs = retriever.invoke(query)

In [ ]:
# Print Retrieved Documents
for i, doc in enumerate(retrieved_docs):
    print("="*80)
    print(f"Document {i+1}")
    print(f"Page: {doc.metadata['page']}")
    print("-"*80)
    print(doc.page_content[:500])
    print("\n")

Document 1
Page: 6
--------------------------------------------------------------------------------
Figure 4. Attendance monitoring system.
The sensors used in this system are referred to as Bluetooth stations (running on Raspberry Pi with
BLE), which are used to measure and collect the RSSI of the students’ devices from the four reference
points to calculate the students’ position in the classroom.
The Bluetooth station node is the BLE hardware that is installed in the classroom, which sends out
Bluetooth signals and it is referenced for position identiﬁcation of a particular student in the c


Document 2
Page: 7
--------------------------------------------------------------------------------
mobile  node  encounters  a  Bluetooth  device  in  the  range,  it  immediately  reads  the  Bluetooth  RSSI 
value  from  the  device.  The  software  in  the  Bluetooth  station  was  developed  based  on  the  Python 
programing language, which Figure 5 illustrates. 
 
Figure 5. Flow chart of

# **🚀 Step 6 — Prompt Engineering**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an expert research assistant.

Answer the user's question ONLY using the provided context.

If the answer is not present in the context, reply:
"I couldn't find that information in the uploaded research paper."

Always be clear and concise.

Context:
{context}

Question:
{input}

Answer:
""")

print("✅ Prompt Created Successfully")

✅ Prompt Created Successfully


# **🚀 Step 7 — Build the Complete RAG Pipeline**




In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [ ]:
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

print(document_chain)

bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nYou are an expert research assistant.\n\nAnswer the user\'s question ONLY using the provided context.\n\nIf the answer is not present in the context, reply:\n"I couldn\'t find that information in the uploaded research paper."\n\nAlways be clear and concise.\n\nContext:\n{context}\n\nQuestion:\n{input}\n\nAnswer:\n'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_

In [ ]:
# Retrieve Documents
query = "What is the main objective of this research paper?"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} documents")

Retrieved 3 documents


In [ ]:
# Inspect Retrieved Documents
for i, doc in enumerate(retrieved_docs):
    print("="*80)
    print(f"Document {i+1}")
    print(f"Page : {doc.metadata['page']}")
    print("-"*80)
    print(doc.page_content[:400])
    print()

Document 1
Page : 6
--------------------------------------------------------------------------------
Figure 4. Attendance monitoring system.
The sensors used in this system are referred to as Bluetooth stations (running on Raspberry Pi with
BLE), which are used to measure and collect the RSSI of the students’ devices from the four reference
points to calculate the students’ position in the classroom.
The Bluetooth station node is the BLE hardware that is installed in the classroom, which sends ou

Document 2
Page : 12
--------------------------------------------------------------------------------
19 1.510 3.949 3.115 44 0.818 4.183 2.703 69 0.460 2.956 2.079
20 0.367 5.148 3.469 45 0.382 2.264 1.342
21 0.510 3.326 1.227 46 0.218 2.258 1.321
22 0.090 1.994 0.849 47 1.457 5.106 2.677
23 0.393 1.782 1.005 48 0.334 4.078 2.221
24 0.507 5.795 1.661 49 1.024 3.335 2.225
The proposed framework was designed and developed with classroom type I before applying on
classroom type II. Fairly good 

In [ ]:
# Run the Document Chain
response = document_chain.invoke({
    "input": query,
    "context": retrieved_docs
})

print(response)

I couldn't find that information in the uploaded research paper.


In [ ]:
def ask_question(question):
    docs = retriever.invoke(question)

    answer = document_chain.invoke({
        "input": question,
        "context": docs
    })

    # Collect page numbers
    pages = sorted(set(doc.metadata.get("page", "Unknown") for doc in docs))

    return {
        "question": question,
        "answer": answer,
        "pages": pages,
        "documents": docs
    }

In [ ]:
def print_result(result):
    print("=" * 80)
    print("QUESTION")
    print(result["question"])

    print("\nANSWER")
    print(result["answer"])

    print("\nSOURCES")
    for page in result["pages"]:
        print(f"📄 Page {page}")

    print("=" * 80)

In [ ]:
result = ask_question("Explain the methodology.")

print_result(result)

QUESTION
Explain the methodology.

ANSWER
The methodology involves using a feedforward multilayer perceptron (FF-MLP) structure with a radial basis function (RBF) network to calculate student positions in the classroom. The experiment was repeated 10 times to measure deviations from the real position of the chairs. Data collection was done using four Bluetooth reference nodes, and at each point, the RSSI was collected by moving the mobile around. The system also uses trilateration or fingerprinting-based approaches to estimate the distance by translating RSSI values.

SOURCES
📄 Page 3
📄 Page 7
📄 Page 8
📄 Page 11
📄 Page 13


In [ ]:
full_text = "\n\n".join([doc.page_content for doc in documents])

print(len(full_text))

75957


In [ ]:
print(full_text[:1000])

information
Article
Classroom Attendance Systems Based on Bluetooth
Low Energy Indoor Positioning Technology for
Smart Campus
Apiruk Puckdeevongs 1,*, N. K. Tripathi 1, Apichon Witayangkurn 1
and Poompat Saengudomlert 2
1 Remote Sensing and Geographic Information Systems Field of Study, School of Engineering and Technology,
Asian Institute of Technology, P .O. Box 4, Klong Luang, Pathumthani 12120, Thailand;
nitinkt@ait.asia (N.K.T.); apichon@ait.asia (A.W.)
2 Telecommunications Field of Study, School of Engineering and Technology, Asian Institute of Technology,
P .O. Box 4, Klong Luang, Pathumthani 12120, Thailand; poompat@gmail.com
* Correspondence: st106421@ait.asia or apiruk.pui@gmail.com; Tel.: +66-86896-3333
Received: 3 May 2020; Accepted: 16 June 2020; Published: 19 June 2020
/gid00030/gid00035/gid00032/gid00030/gid00038/gid00001/gid00033/gid00042/gid00045/gid00001
/gid00048/gid00043/gid00031/gid00028/gid00047/gid00032/gid00046
Abstract: Student attendance during classroom hours

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chunk_summary_prompt = ChatPromptTemplate.from_template("""
You are an expert research paper analyst.

Summarize the following part of the research paper.

Focus on:
- Main idea
- Important findings
- Key technical details

Text:
{text}
""")

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chunk_summary_chain = (
    chunk_summary_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
chunk_summaries = []

for i, chunk in enumerate(chunks):
    print(f"Summarizing chunk {i+1}/{len(chunks)}")

    summary = chunk_summary_chain.invoke({
        "text": chunk.page_content
    })

    chunk_summaries.append(summary)

Summarizing chunk 1/100
Summarizing chunk 2/100
Summarizing chunk 3/100
Summarizing chunk 4/100
Summarizing chunk 5/100
Summarizing chunk 6/100
Summarizing chunk 7/100
Summarizing chunk 8/100
Summarizing chunk 9/100
Summarizing chunk 10/100
Summarizing chunk 11/100
Summarizing chunk 12/100
Summarizing chunk 13/100
Summarizing chunk 14/100
Summarizing chunk 15/100
Summarizing chunk 16/100
Summarizing chunk 17/100
Summarizing chunk 18/100
Summarizing chunk 19/100
Summarizing chunk 20/100
Summarizing chunk 21/100
Summarizing chunk 22/100
Summarizing chunk 23/100
Summarizing chunk 24/100
Summarizing chunk 25/100
Summarizing chunk 26/100
Summarizing chunk 27/100
Summarizing chunk 28/100
Summarizing chunk 29/100
Summarizing chunk 30/100
Summarizing chunk 31/100
Summarizing chunk 32/100
Summarizing chunk 33/100
Summarizing chunk 34/100
Summarizing chunk 35/100
Summarizing chunk 36/100
Summarizing chunk 37/100
Summarizing chunk 38/100
Summarizing chunk 39/100
Summarizing chunk 40/100
Summarizi

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kvhntwy1fe9vptmkevb0gvye` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99307, Requested 811. Please try again in 1m41.952s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
combined_summary = "\n\n".join(chunk_summaries)

print(combined_summary[:2000])

Here is a summary of the provided text, focusing on the main idea, important findings, and key technical details:

**Main Idea:**
The main idea of the research paper is to explore the development of a classroom attendance system using Bluetooth Low Energy (BLE) indoor positioning technology for a smart campus.

**Important Findings:**
Unfortunately, the provided text does not reveal any specific findings, as it appears to be the introductory or metadata section of the research paper. However, it can be inferred that the paper will discuss the design, implementation, and evaluation of a BLE-based attendance system.

**Key Technical Details:**
Some key technical details that can be gathered from the text include:

* The use of Bluetooth Low Energy (BLE) technology for indoor positioning.
* The application of the system in a smart campus setting.
* The involvement of researchers from the Remote Sensing and Geographic Information Systems Field of Study, as well as the Telecommunications Fi

In [ ]:
final_summary_prompt = ChatPromptTemplate.from_template("""
You are an expert research analyst.

Using the following chunk summaries, generate an Executive Summary.

Include:

1. Research Objective
2. Problem Statement
3. Methodology
4. Experimental Results
5. Conclusion

Chunk Summaries:
{text}
""")

In [ ]:
final_summary_chain = (
    final_summary_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
executive_summary = final_summary_chain.invoke({
    "text": combined_summary
})

print(executive_summary)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kvhntwy1fe9vptmkevb0gvye` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99109, Requested 16228. Please try again in 3h40m51.168s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}